[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/VaR_CEE_FM/blob/main/VaR_CEE_DataDownload/VaR_CEE_DataDownload.ipynb)

# VaR_CEE_DataDownload

Downloads daily stock index data from Stooq and FX exchange rate data from Yahoo Finance
for five CEE countries (Romania, Poland, Czechia, Hungary, Bulgaria).
Includes fallback mechanisms via `pandas_datareader` and `yfinance`.

**Published in:** *Can Foundation Models Manage Risk? Zero-Shot VaR and ES Forecasting for CEE Markets*

**Author:** Daniel Traian Pele

In [ ]:
# Install dependencies (Colab)
!pip install -q yfinance pandas_datareader

import time
import json
import urllib.request
import urllib.error
import datetime as dt
import numpy as np
import pandas as pd
from pathlib import Path
from io import StringIO

# ── Configuration (inline for Colab) ───────────────────────
MARKETS = {
    "Romania":  {"index": "BET",   "stooq": "^BET",   "fx": "EUR/RON", "yahoo_fx": "EURRON=X"},
    "Poland":   {"index": "WIG20", "stooq": "^WIG20", "fx": "EUR/PLN", "yahoo_fx": "EURPLN=X"},
    "Czechia":  {"index": "PX",    "stooq": "^PX",    "fx": "EUR/CZK", "yahoo_fx": "EURCZK=X"},
    "Hungary":  {"index": "BUX",   "stooq": "^BUX",   "fx": "EUR/HUF", "yahoo_fx": "EURHUF=X"},
    "Bulgaria": {"index": "SOFIX", "stooq": "^SOFIX", "fx": "USD/BGN", "yahoo_fx": "USDBGN=X"},
}
START_DATE = "2007-01-01"
END_DATE = "2025-12-31"

RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def download_stooq(ticker, start, end, max_retries=3):
    """Download daily data from Stooq via CSV URL."""
    d1 = start.replace("-", "")
    d2 = end.replace("-", "")
    sym = ticker.replace("^", "").lower()
    url = f"https://stooq.com/q/d/l/?s={sym}&d1={d1}&d2={d2}&i=d"
    print(f"  Stooq URL: {url}")
    for attempt in range(max_retries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=30) as resp:
                data = resp.read().decode("utf-8")
            if "No data" in data or len(data.strip()) < 50:
                print(f"  Warning: No data returned for {ticker}")
                return None
            df = pd.read_csv(StringIO(data))
            if "Date" in df.columns:
                df["Date"] = pd.to_datetime(df["Date"])
                df = df.set_index("Date").sort_index()
            elif "date" in df.columns:
                df["date"] = pd.to_datetime(df["date"])
                df = df.set_index("date").sort_index()
            print(f"  Downloaded {len(df)} rows for {ticker}")
            return df
        except Exception as e:
            print(f"  Attempt {attempt+1} failed for {ticker}: {e}")
            time.sleep(2 * (attempt + 1))
    return None

In [ ]:
def download_stooq_pandas_datareader(ticker, start, end):
    """Fallback: use pandas_datareader."""
    try:
        from pandas_datareader import data as pdr
        df = pdr.DataReader(ticker, "stooq", start=start, end=end)
        df = df.sort_index()
        print(f"  Downloaded {len(df)} rows for {ticker} via pandas_datareader")
        return df
    except Exception as e:
        print(f"  pandas_datareader failed for {ticker}: {e}")
        return None

In [ ]:
def download_yahoo_fx(ticker, start, end, max_retries=3):
    """Download FX data from Yahoo Finance v8 API."""
    start_ts = int(dt.datetime.strptime(start, "%Y-%m-%d").timestamp())
    end_ts = int(dt.datetime.strptime(end, "%Y-%m-%d").timestamp())
    encoded = ticker.replace("=", "%3D")
    url = (f"https://query1.finance.yahoo.com/v8/finance/chart/{encoded}"
           f"?period1={start_ts}&period2={end_ts}&interval=1d")
    print(f"  Yahoo URL: {url[:80]}...")
    for attempt in range(max_retries):
        try:
            req = urllib.request.Request(url, headers={
                "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)"
            })
            with urllib.request.urlopen(req, timeout=30) as resp:
                raw = json.loads(resp.read().decode("utf-8"))
            result = raw["chart"]["result"][0]
            timestamps = result["timestamp"]
            closes = result["indicators"]["quote"][0]["close"]
            dates = [dt.datetime.utcfromtimestamp(ts).strftime("%Y-%m-%d") for ts in timestamps]
            df = pd.DataFrame({"Date": dates, "Close": closes})
            df["Date"] = pd.to_datetime(df["Date"])
            df = df.set_index("Date").sort_index()
            df = df.dropna()
            print(f"  Downloaded {len(df)} rows for {ticker}")
            return df
        except Exception as e:
            print(f"  Attempt {attempt+1} failed for {ticker}: {e}")
            time.sleep(2 * (attempt + 1))
    return None

In [ ]:
def download_yahoo_fx_yfinance(ticker, start, end):
    """Fallback: use yfinance."""
    try:
        import yfinance as yf
        df = yf.download(ticker, start=start, end=end, progress=False)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df = df[["Close"]].dropna()
        print(f"  Downloaded {len(df)} rows for {ticker} via yfinance")
        return df
    except Exception as e:
        print(f"  yfinance failed for {ticker}: {e}")
        return None

In [ ]:
# Main download loop
print("=" * 60)
print("DOWNLOADING DATA")
print("=" * 60)

for country, info in MARKETS.items():
    print(f"\n--- {country} ---")

    # Download stock index
    idx_file = RAW_DIR / f"{info['index']}.csv"
    if idx_file.exists():
        print(f"  {info['index']} already downloaded, skipping")
    else:
        print(f"  Downloading {info['index']} from Stooq...")
        df = download_stooq(info['stooq'], START_DATE, END_DATE)
        if df is None:
            print(f"  Trying pandas_datareader fallback...")
            df = download_stooq_pandas_datareader(info['stooq'], START_DATE, END_DATE)
        if df is not None:
            df.to_csv(idx_file)
            print(f"  Saved {idx_file}")
        else:
            print(f"  ERROR: Could not download {info['index']}")

    # Download FX
    fx_name = info['fx'].replace("/", "")
    fx_file = RAW_DIR / f"{fx_name}.csv"
    if fx_file.exists():
        print(f"  {fx_name} already downloaded, skipping")
    else:
        print(f"  Downloading {info['yahoo_fx']} from Yahoo Finance...")
        df = download_yahoo_fx(info['yahoo_fx'], START_DATE, END_DATE)
        if df is None:
            print(f"  Trying yfinance fallback...")
            df = download_yahoo_fx_yfinance(info['yahoo_fx'], START_DATE, END_DATE)
        if df is not None:
            df.to_csv(fx_file)
            print(f"  Saved {fx_file}")
        else:
            print(f"  ERROR: Could not download {fx_name}")

print("\n=== Download complete ===")

In [ ]:
# Download all raw CSVs as zip
from google.colab import files
import zipfile, io

csv_files = list(RAW_DIR.glob("*.csv"))
if csv_files:
    zip_buffer = io.BytesIO()
    with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in csv_files:
            zf.write(f, f.name)
    zip_buffer.seek(0)
    with open("raw_data.zip", "wb") as fout:
        fout.write(zip_buffer.read())
    files.download("raw_data.zip")
    print(f"Downloaded {len(csv_files)} CSV files as raw_data.zip")